In [ ]:
!pip install pydicom

In [ ]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import pydicom

logger = logging.getLogger(__name__)

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the drive containing the images
    drive.mount('/content/drive')

    # List contents (IPython magic command)
    !ls

In [ ]:
if IN_COLAB:
    # Change the working to the current notebook's directory
    %cd /content/drive/MyDrive/
else:
    # Ask the user to fill in the base data folder path
    base_data_folder = input("Please enter the path to the base data folder path: ")
    os.chdir(base_data_folder)

## CAMUS release — folder layout

The released CAMUS dataset lives under `<data_folder>/release/camus/`. All images and masks are stored as 8-bit Secondary Capture DICOM files (`.dcm`, Modality = `US`, `PhotometricInterpretation = MONOCHROME2`, `BitsStored = 8`, slope = 1, intercept = 0). Non-pixel metadata lives in standard DICOM tags plus the `LabBioimmagini v1` private block at group `0x0035`.

```
camus/
├── train/
│   ├── clean/
│   │   ├── patientXXXX_2CH_ED.dcm   # clean ED frame, 2-chamber view
│   │   ├── patientXXXX_2CH_ES.dcm   # clean ES frame, 2-chamber view
│   │   ├── patientXXXX_4CH_ED.dcm   # clean ED frame, 4-chamber view
│   │   ├── patientXXXX_4CH_ES.dcm   # clean ES frame, 4-chamber view
│   │   └── ...
│   ├── masks/
│   │   ├── patientXXXX_2CH_ED.dcm   # LV / Myo / LA mask, same frame_id as the clean file
│   │   └── ...
│   └── perturbed/
│       ├── patientXXXX_2CH_ED_d0.dcm   # perturbation of the matching clean frame
│       └── ...
├── val/
│   └── ...
├── test/
│   └── ...
└── index.csv   # routing CSV: perturbed_id, clean_id, mask_id, split
```

Each perturbed DICOM is a photometric alteration of its matching `clean_id` — the mask stays valid for both because every perturbation preserves geometry. This is for example what can be used to train a DL denoiser: the perturbed frame is the input, the clean frame is the target.

### Mask Segmentation Labels

The segmentation masks use the following values to identify distinct anatomical structures:

| Value | Structure |
|---|---|
| 0 | Background |
| 1 | Left Ventricle (LV) |
| 2 | Myocardium (Myo) |
| 3 | Left Atrium (LA) |

Because this project focuses specifically on predicting the volume of the left ventricle cavity, **Label 1 (Left Ventricle)** is the primary target relevant to our analysis. The other anatomical labels (Myocardium and Left Atrium) are included in the dataset for completeness but are not utilized for this specific volume estimation task.


## Visualisation

In [ ]:
# ── Visualization parameters ─────────────────────────────────────────────────

# Structures to overlay on the echo images.
# Pick any subset of: 1 (LV), 2 (Myocardium), 3 (Left Atrium).
LABELS_TO_SHOW = [1, 2, 3]

# RGB colours assigned to each label (values in [0, 1])
LABEL_COLORS = {
    1: (1.00, 0.20, 0.20),  # LV  → red
    2: (0.20, 1.00, 0.20),  # Myo → green
    3: (0.20, 0.80, 1.00),  # LA  → cyan
}
LABEL_NAMES = {1: "LV", 2: "Myo", 3: "LA"}

# Transparency of the coloured overlay (0 = invisible, 1 = opaque)
OVERLAY_ALPHA = 0.45

In [ ]:
DATA_FOLDER_PATH = Path(
    "./release"
)
assert Path(
    DATA_FOLDER_PATH
).exists(), f"DATA_FOLDER_PATH does not exist: {DATA_FOLDER_PATH}"
for file in Path(DATA_FOLDER_PATH).iterdir():
    print(f"Found file: {file.name}")

In [ ]:
# ── Shared helpers ──────────────────────────────────────────────────────────
# `read_release_dicom` loads a released DICOM (uint8 MONOCHROME2) and extracts
# both the standard tags we care about and every element present in the
# `LabBioimmagini v1` private block at group 0x0035. Offsets below mirror
# `src/io/_dicom_release.py::_PRIVATE_ELEMENT_MAP`.
PRIVATE_GROUP = 0x0035
PRIVATE_CREATOR = "LabBioimmagini v1"
PRIVATE_OFFSETS = {
    0x01: "dataset",
    0x02: "split",
    0x03: "kind",
    0x04: "view",
    0x05: "phase",
    0x06: "quality",  # CAMUS only
    0x07: "draw",
    0x08: "machine",  # may be absent
    0x09: "exam_id",  # inhouse only
    0x10: "EDV_LV_ml",  # inhouse only
    0x11: "EDV_LV_label",  # inhouse only
    0x20: "group_id",  # inhouse grouped mode
}


def read_release_dicom(path):
    """Return `(image_uint8, metadata_dict)` for a released DICOM file.

    The metadata dict carries selected standard tags plus every private
    element that is actually present in the file (missing offsets are
    silently skipped — not every field applies to every dataset).
    """
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array
    if img.ndim == 3:
        img = img[0]

    meta = {
        "PatientID": getattr(ds, "PatientID", None),
        "Modality": getattr(ds, "Modality", None),
        "Rows": int(ds.Rows),
        "Columns": int(ds.Columns),
        "PixelSpacing": [float(v) for v in getattr(ds, "PixelSpacing", [None, None])],
        "PhotometricInterpretation": getattr(ds, "PhotometricInterpretation", None),
        "BitsStored": int(getattr(ds, "BitsStored", 8)),
        "SOPInstanceUID": str(getattr(ds, "SOPInstanceUID", "")),
    }
    try:
        block = ds.private_block(PRIVATE_GROUP, PRIVATE_CREATOR)
    except KeyError:
        return img, meta
    for offset, key in PRIVATE_OFFSETS.items():
        try:
            meta[key] = block[offset].value
        except KeyError:
            continue
    return img, meta


def build_overlay(image_u8, mask_u8, labels, colors, alpha):
    """Alpha-blend coloured mask labels onto a grayscale image.

    `image_u8` is `(H, W)` uint8; `mask_u8` is `(H, W)` with values in
    `{0, 1, 2, 3}`. Returns `(H, W, 3)` float in [0, 1].
    """
    base = np.clip(image_u8.astype(np.float32) / 255.0, 0.0, 1.0)
    rgb = np.stack([base, base, base], axis=-1)
    for lab in labels:
        sel = mask_u8 == lab
        if not np.any(sel):
            continue
        c = np.asarray(colors[lab], dtype=np.float32)
        rgb[sel] = (1.0 - alpha) * rgb[sel] + alpha * c
    return np.clip(rgb, 0.0, 1.0)


def legend_patches(labels, colors, names):
    """Matplotlib legend handles for the overlay colours."""
    return [
        Patch(facecolor=colors[lab], edgecolor="black", label=names[lab])
        for lab in labels
    ]

In [ ]:
# ── Pick a CAMUS patient / split / perturbation draw ───────────────────────
# `split` must be one of {"train", "val", "test"}. `draw` selects which
# perturbation of the clean frame to pair with (0-indexed; 0 always exists).
split = "train"
patient_id = 1
draw = 0

patient_name = f"patient{patient_id:04d}"

camus_split_root = DATA_FOLDER_PATH / "camus" / split
clean_root = camus_split_root / "clean"
mask_root = camus_split_root / "masks"
perturbed_root = camus_split_root / "perturbed"

for p in (clean_root, mask_root, perturbed_root):
    assert p.exists(), f"Missing release subfolder: {p}"

print(f"split         : {split}")
print(f"patient       : {patient_name}")
print(f"draw          : d{draw}")
print(f"clean dir     : {clean_root}")
print(f"mask dir      : {mask_root}")
print(f"perturbed dir : {perturbed_root}")

In [ ]:
# ── Load the four keyframes (2CH/4CH × ED/ES): clean + mask + perturbed ───
# For each (view, phase) the clean frame, its mask, and one perturbation draw
# share the same `frame_id`. The perturbed file name adds a `_d<draw>` suffix.
frames: dict[tuple[str, str], dict] = {}
for view in ("2CH", "4CH"):
    for phase in ("ED", "ES"):
        fid = f"{patient_name}_{view}_{phase}"
        clean_img, clean_meta = read_release_dicom(clean_root / f"{fid}.dcm")
        mask_img, _mask_meta = read_release_dicom(mask_root / f"{fid}.dcm")
        pert_img, pert_meta = read_release_dicom(perturbed_root / f"{fid}_d{draw}.dcm")
        frames[(view, phase)] = {
            "frame_id": fid,
            "clean": clean_img,
            "mask": mask_img,
            "perturbed": pert_img,
            "meta": clean_meta,
            "pmeta": pert_meta,
        }

print(f"Loaded {len(frames)} keyframes for {patient_name} ({split}, draw d{draw}).")

In [ ]:
# ── Clean keyframes: 2×2 grid (rows = view, cols = phase) ──────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
for row, view in enumerate(("2CH", "4CH")):
    for col, phase in enumerate(("ED", "ES")):
        f = frames[(view, phase)]
        meta = f["meta"]
        axes[row, col].imshow(f["clean"], cmap="gray")
        axes[row, col].set_title(
            f"{view} – {phase}  |  quality: {meta.get('quality', '—')}",
            fontsize=10,
        )
        axes[row, col].axis("off")
fig.suptitle(
    f"{patient_name} — clean keyframes ({split})", fontsize=12, fontweight="bold"
)
plt.show()

In [ ]:
header = f"{'(view, phase)':<16} {'size':<14} {'spacing [mm/px]':<22} {'quality':<10} {'kind':<10} {'draw':<6}"
print(header)
print("-" * len(header))
for view in ("2CH", "4CH"):
    for phase in ("ED", "ES"):
        f = frames[(view, phase)]
        m = f["meta"]
        pm = f["pmeta"]
        sp = m["PixelSpacing"]
        sp_str = f"[{sp[0]:.3f}, {sp[1]:.3f}]" if sp[0] is not None else "—"
        print(
            f"{str((view, phase)):<16} "
            f"{m['Rows']}x{m['Columns']:<10} "
            f"{sp_str:<22} "
            f"{str(m.get('quality', '—')):<10} "
            f"{str(m.get('kind', '—')):<10} "
            f"clean=0 / pert={pm.get('draw', '—')}"
        )
print()
print(f"PatientID (CAMUS)        : {frames[('2CH', 'ED')]['meta']['PatientID']}")
print(
    f"Dataset / split (private): {frames[('2CH', 'ED')]['meta'].get('dataset')} / {frames[('2CH', 'ED')]['meta'].get('split')}"
)

## Keyframe visualisation

Each row shows one keyframe: the **raw echo image** (left) and the same image with the selected
segmentation labels overlaid (right).
Rows: 2-chamber End-Diastole · 2-chamber End-Systole · 4-chamber End-Diastole · 4-chamber End-Systole.


In [ ]:
# ── Keyframe overlay figure ─────────────────────────────────────────────────
keyframes = [
    ("2CH", "ED"),
    ("2CH", "ES"),
    ("4CH", "ED"),
    ("4CH", "ES"),
]

fig, axes = plt.subplots(
    len(keyframes), 2, figsize=(10, 4.5 * len(keyframes)), constrained_layout=True
)

for row, (view, phase) in enumerate(keyframes):
    f = frames[(view, phase)]
    img = f["clean"]
    mask = f["mask"]
    meta = f["meta"]
    sp = meta["PixelSpacing"]
    overlay = build_overlay(img, mask, LABELS_TO_SHOW, LABEL_COLORS, OVERLAY_ALPHA)

    ax_raw, ax_ov = axes[row]

    ax_raw.imshow(img, cmap="gray", aspect="equal")
    ax_raw.set_title(f"{view} – {phase}  |  clean echo", fontsize=10)
    ax_raw.axis("off")

    ax_ov.imshow(overlay, aspect="equal")
    ax_ov.set_title(
        f"{view} – {phase}  |  overlay  "
        f"[{patient_name}  quality: {meta.get('quality', '—')}  "
        f"spacing: {sp[0]:.3f} mm/px]",
        fontsize=8,
    )
    ax_ov.axis("off")
    ax_ov.legend(
        handles=legend_patches(LABELS_TO_SHOW, LABEL_COLORS, LABEL_NAMES),
        loc="lower right",
        fontsize=7,
        framealpha=0.7,
    )

q2 = frames[("2CH", "ED")]["meta"].get("quality", "—")
q4 = frames[("4CH", "ED")]["meta"].get("quality", "—")
fig.suptitle(
    f"{patient_name} — apical views at ED and ES  (quality: 2CH={q2}, 4CH={q4})",
    fontsize=11,
    fontweight="bold",
)
plt.show()

## Clean vs. perturbed — denoiser training pairs

For the Deep Learning denoiser training on `(perturbed → clean)` pairs: the perturbed frame is the input, the clean frame is the target, and the mask is unchanged for both because every perturbation in this release is **photometric** (speckle, contrast, blur, …) — it never moves pixels around. The matching file for `clean/<frame_id>.dcm` is `perturbed/<frame_id>_d<draw>.dcm`; higher-draw files are additional independent perturbations of the *same* clean frame.


In [ ]:
# ── Clean vs. perturbed (denoiser training pair) ───────────────────────────
# For every clean frame in `release/camus/<split>/clean/<frame_id>.dcm` there is
# at least one matching perturbed file at
# `release/camus/<split>/perturbed/<frame_id>_d<draw>.dcm`. A DL denoiser trains
# on `(perturbed → clean)` pairs; the two keyframes below illustrate that pair
# at both apical views (ED only, to keep the figure compact).
pair_views = [("2CH", "ED"), ("4CH", "ED")]

fig, axes = plt.subplots(
    len(pair_views), 2, figsize=(11, 5.5 * len(pair_views)), constrained_layout=True
)
if len(pair_views) == 1:
    axes = axes[None, :]

for row, (view, phase) in enumerate(pair_views):
    f = frames[(view, phase)]
    clean = f["clean"]
    pert = f["perturbed"]
    diff = np.abs(clean.astype(np.int16) - pert.astype(np.int16))

    axes[row, 0].imshow(clean, cmap="gray", vmin=0, vmax=255)
    axes[row, 0].set_title(
        f"{view} – {phase}  |  clean  ({f['frame_id']}.dcm)", fontsize=9
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(pert, cmap="gray", vmin=0, vmax=255)
    axes[row, 1].set_title(
        f"{view} – {phase}  |  perturbed  ({f['frame_id']}_d{draw}.dcm)\n"
        f"|clean − perturbed|  mean={diff.mean():.2f}  max={int(diff.max())}",
        fontsize=9,
    )
    axes[row, 1].axis("off")

fig.suptitle(
    f"{patient_name} — denoiser input/target pairs  (clean ⇆ perturbed d{draw})",
    fontsize=11,
    fontweight="bold",
)
plt.show()

# Sanity check: a well-formed release contains non-trivial perturbations.
first_clean = frames[("2CH", "ED")]["clean"]
first_pert = frames[("2CH", "ED")]["perturbed"]
assert first_clean.shape == first_pert.shape, "clean and perturbed shapes diverged"
assert np.any(
    first_clean != first_pert
), "perturbed frame is bit-identical to clean — unexpected"
print(f"✓ clean vs. perturbed differ for {patient_name} 2CH-ED (d{draw}).")

---
## In-house dataset

The in-house dataset ships as **perturbed uint8 DICOMs** under `<data_folder>/release/inhouse/<split>/<image_id>.dcm`, with a routing CSV `release/inhouse/labels.csv` listing every `image_id` and its split assignment. It is **ED-only** and only two apical views are present — `2CH` (A2C on the raw side) and `4CH` (A4C). Unlike CAMUS there are no clean frames and no segmentation masks in the release. It is required that students apply their denoising strategy (i.e., either a traditional image processing technique or a deep learning model) to the perturbed image and then solve one of the two downstream tasks below.

| Task | Private tag | Metric |
|---|---|---|
| `volume_regression` | `EDV_LV_ml` (float, mL) | MAE and R² |
| `volume_binary` | `EDV_LV_label` (`low/normal` → 0, `high/dilated` → 1) | balanced accuracy |

Every task label is embedded **inside each DICOM** via the `LabBioimmagini v1` private block (group `0x0035`).

### Filename convention

```
release/inhouse/<split>/exam<exam_id>_<view>_ED_d<draw>.dcm
```

- `<exam_id>` matches the `id_exam` column on the raw side (instructor-side only).
- `<view>` is `2CH` or `4CH`.
- `<draw>` is the perturbation draw index (0, 1, …).


In [ ]:
# ── Inhouse release path setup ─────────────────────────────────────────────
INHOUSE_RELEASE = DATA_FOLDER_PATH / "inhouse"
INHOUSE_LABELS_CSV = INHOUSE_RELEASE / "labels.csv"

assert INHOUSE_RELEASE.exists(), f"Inhouse release root not found: {INHOUSE_RELEASE}"
assert INHOUSE_LABELS_CSV.exists(), f"labels.csv not found: {INHOUSE_LABELS_CSV}"

print(f"Inhouse release : {INHOUSE_RELEASE}")
print(f"labels.csv      : {INHOUSE_LABELS_CSV}")
print(
    f"Splits on disk  : {sorted(p.name for p in INHOUSE_RELEASE.iterdir() if p.is_dir())}"
)

In [ ]:
# ── Load the routing CSV and peek at its structure ─────────────────────────
df_labels = pd.read_csv(INHOUSE_LABELS_CSV)
assert len(df_labels) > 0, "labels.csv is empty — check that the release was generated."
assert set(df_labels.columns) == {
    "image_id",
    "split",
}, f"Unexpected columns in labels.csv: {list(df_labels.columns)}"

print(f"Shape          : {df_labels.shape}  (rows x columns)")
print(f"Columns        : {list(df_labels.columns)}")
print()
print("Split counts:")
print(df_labels["split"].value_counts().to_string())
print()

# Filenames follow `exam<exam_id>_<view>_<phase>_d<draw>`. The regex below
# makes that structure explicit so students see how to parse an image_id.
parsed = df_labels["image_id"].str.extract(
    r"^exam(?P<exam_id>\d+)_(?P<view>2CH|4CH)_(?P<phase>ED|ES)_d(?P<draw>\d+)$"
)
print("Parsed filename fields (head):")
print(
    pd.concat(
        [df_labels[["image_id", "split"]].head(), parsed.head()], axis=1
    ).to_string(index=False)
)

---
## DICOM loading and private metadata (inhouse release)

Every released inhouse DICOM is a **Secondary Capture, uint8, MONOCHROME2** image. That means `ds.pixel_array` is already displayable (i.e., no `RescaleSlope`/`RescaleIntercept` application and no `MONOCHROME1` inversion is required, contrary to what was needed on the raw side).

Non-pixel information lives in two places:

- a handful of **standard DICOM tags** (`Modality`, `Rows`, `Columns`, `PixelSpacing`, `PhotometricInterpretation`, `BitsStored`, `PatientID`, `SOPInstanceUID`);
- the project-specific **`LabBioimmagini v1` private block at group `0x0035`**, which carries the per-image task labels needed to supervise `volume_regression` and `volume_binary`.

### Private block offsets

Mirrored from `src/io/_dicom_release.py::_PRIVATE_ELEMENT_MAP`.

| Offset | Keyword | VR | Used by |
|---:|---|---|---|
| `0x01` | `dataset` | LO | both |
| `0x02` | `split` | LO | both |
| `0x03` | `kind` | LO | both (`clean` / `perturbed` / `mask`) |
| `0x04` | `view` | LO | both (`2CH` / `4CH`) |
| `0x05` | `phase` | LO | both (`ED` / `ES`) |
| `0x07` | `draw` | IS | both (perturbation draw index) |
| `0x09` | `exam_id` | LO | inhouse |
| **`0x10`** | **`EDV_LV_ml`** | **DS** | **inhouse — `volume_regression`** |
| **`0x11`** | **`EDV_LV_label`** | **LO** | **inhouse — `volume_binary`** |
| `0x20` | `group_id` | LO | inhouse grouped-sampling mode |

Read the block with:

```python
block = ds.private_block(0x0035, "LabBioimmagini v1")
edv_ml = float(block[0x10].value)
```

`block[off].value` raises `KeyError` for any offset the writer did not populate — the helper `_priv(block, off)` below handles this gracefully.


In [ ]:
# ── Load one released inhouse DICOM and print its attributes ────────────────
IMAGE_ID = df_labels.iloc[0]["image_id"]  # change to any image_id in labels.csv
image_split = df_labels.loc[df_labels["image_id"] == IMAGE_ID, "split"].iloc[0]
dcm_path = INHOUSE_RELEASE / image_split / f"{IMAGE_ID}.dcm"
assert dcm_path.exists(), f"DICOM not found: {dcm_path}"

ds = pydicom.dcmread(str(dcm_path))

# Release files are uint8 MONOCHROME2 — the following assertions guard against
# accidentally pointing this notebook at a raw (non-release) DICOM.
assert (
    str(ds.PhotometricInterpretation).upper() == "MONOCHROME2"
), f"Expected MONOCHROME2, got {ds.PhotometricInterpretation}"
assert int(ds.BitsStored) == 8, f"Expected BitsStored=8, got {ds.BitsStored}"

block = ds.private_block(PRIVATE_GROUP, PRIVATE_CREATOR)


def _priv(block, offset, default="—"):
    """Return `block[offset].value` or `default` if the offset is missing."""
    try:
        return block[offset].value
    except KeyError:
        return default


print("=== File ===")
print(f"  path            : {dcm_path}")
print(f"  SOPInstanceUID  : {ds.SOPInstanceUID}")
print(f"  PatientID       : {ds.PatientID}")
print()
print("=== Standard DICOM ===")
print(f"  Modality                 : {ds.Modality}")
print(f"  Rows x Columns           : {ds.Rows} x {ds.Columns}")
print(f"  PixelSpacing (mm/px)     : {list(ds.PixelSpacing)}")
print(f"  PhotometricInterpretation: {ds.PhotometricInterpretation}")
print(f"  BitsAllocated / Stored   : {ds.BitsAllocated} / {ds.BitsStored}")
print()
print("=== Release metadata (LabBioimmagini v1, group 0x0035) ===")
print(f"  dataset  : {_priv(block, 0x01)}")
print(f"  split    : {_priv(block, 0x02)}")
print(f"  kind     : {_priv(block, 0x03)}")
print(f"  view     : {_priv(block, 0x04)}")
print(f"  phase    : {_priv(block, 0x05)}")
print(f"  draw     : {_priv(block, 0x07)}")
print(f"  exam_id  : {_priv(block, 0x09)}")
print()
print("=== Task labels ===")
edv_ml = _priv(block, 0x10)
edv_lbl = _priv(block, 0x11)
print(f"  volume_regression  →  EDV_LV_ml     : {edv_ml}  (mL)")
print(
    f"  volume_binary      →  EDV_LV_label  : {edv_lbl}  (low/normal → 0, high/dilated → 1)"
)
print()
print("=== Optional ===")
print(f"  group_id : {_priv(block, 0x20)}  (grouped-sampling mode only)")
print(f"  machine  : {_priv(block, 0x08)}  (may be absent in current releases)")

### Visualising the image

`ds.pixel_array` is already a displayable uint8 array (0 = black, 255 = white). No calibration step is needed.


In [ ]:
# ── Display the loaded inhouse image ───────────────────────────────────────
pixels = ds.pixel_array
if pixels.ndim == 3:
    pixels = pixels[0]
sp = list(ds.PixelSpacing)

fig, ax = plt.subplots(1, 1, figsize=(7, 7), constrained_layout=True)
ax.imshow(pixels, cmap="gray", vmin=0, vmax=255, aspect="equal")
ax.axis("off")
ax.set_title(
    f"{IMAGE_ID}  |  split={image_split}  "
    f"view={_priv(block, 0x04)}  phase={_priv(block, 0x05)}  "
    f"exam_id={_priv(block, 0x09)}\n"
    f"size: {pixels.shape[0]}x{pixels.shape[1]} px   "
    f"spacing: {sp[0]:.3f} x {sp[1]:.3f} mm/px\n"
    f"EDV_LV_ml={_priv(block, 0x10)}   "
    f"EDV_LV_label={_priv(block, 0x11)}   ",
    fontsize=9,
)
plt.show()